# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/row-huh/ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

`avg_position` has **1,205 rows sitting at exactly 0**, which the data dictionary flags as "no data," not rank zero — those get dropped before any position-based math from here on (30,000 → 28,795 rows). `impressions_90d` is heavily right-skewed: median 731 but a max of 517,715, so raw impressions will dominate any score that multiplies by it unless I'm deliberate about that. `ctr` is mostly near-zero (median 0.07, i.e. 0.07%) with a long tail out to 100. And `scroll_rate`/`ai_traffic_pct` both go **above 100** (119 and 23 rows respectively) — not a bug, their numerator/denominator come from different measurement systems, so I'm treating those as legitimate outliers, not something to clip.

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
qualified = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].copy()

print("avg_position == 0 (no data, not rank zero):", (qualified.avg_position == 0).sum())
q = qualified[qualified.avg_position > 0].copy()
print("rows after dropping avg_position==0:", len(q))
print()
print(qualified.impressions_90d.describe())
print()
print(qualified[["ctr", "scroll_rate", "ai_traffic_pct"]].describe())
print("scroll_rate > 100:", (qualified.scroll_rate > 100).sum(), "| ai_traffic_pct > 100:", (qualified.ai_traffic_pct > 100).sum())


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Test 1 — "Better position gets higher CTR."** `corr(avg_position, ctr) = -0.08` (negative is the right direction: worse position number → lower CTR). Direction holds but it's weak. **Verdict: MIXED** — real, but far from the whole story; something else is doing most of the work.

**Test 2 — "Longer content gets more engagement."** `corr(word_count, scroll_rate) = -0.148`, `corr(word_count, engagement_rate) = -0.069` — both **negative**, opposite of the hypothesis. Longer pages don't scroll or engage better here, if anything slightly worse. **Verdict: OPPOSITE.**

**Test 3 — "Staler content (longer since last update) gets lower CTR."** `corr(days_since_last_update, ctr) = -0.026` — essentially zero. **Verdict: FALSE** — staleness alone doesn't predict CTR in this data; whatever a `days_since_last_update` threshold is doing in a rule, it isn't doing it through CTR.

In [ ]:
# Test 1
print("corr(avg_position, ctr):", round(q[["avg_position", "ctr"]].corr().iloc[0, 1], 3))

# Test 2
w = q.dropna(subset=["word_count", "scroll_rate", "engagement_rate"])
print("corr(word_count, scroll_rate):", round(w[["word_count", "scroll_rate"]].corr().iloc[0, 1], 3))
print("corr(word_count, engagement_rate):", round(w[["word_count", "engagement_rate"]].corr().iloc[0, 1], 3))

# Test 3
print("corr(days_since_last_update, ctr):", round(q[["days_since_last_update", "ctr"]].corr().iloc[0, 1], 3))


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The guide's `declining_with_demand` reason code fires on `trend_direction == "down"` **and** `impressions_90d >= 100` — its implicit assumption is that a real decline is one backed by actual traffic, not noise on a near-zero-traffic page. Checking that: among pages already trending down, **80.9%** clear the ≥100-impression bar, vs. **73.4%** across all qualifying pages. So decline is, if anything, *more* concentrated among pages with real demand behind them, not less — the flag's assumption holds up rather than being an arbitrary cutoff that excludes most of the group it's meant to catch.

In [ ]:
down = qualified[qualified.trend_direction == "down"]
print("trend_direction == 'down' rows:", len(down))
print("pct of those with impressions_90d >= 100:", round((down.impressions_90d >= 100).mean() * 100, 1))
print("pct of ALL qualifying rows with impressions_90d >= 100:", round((qualified.impressions_90d >= 100).mean() * 100, 1))


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Position alone is a weak CTR predictor and word count buys nothing — so single-signal rules ("it's long, it's ranked well, ship it") aren't well supported by this data, and a scoring approach that weighs several weak signals together will genuinely outperform any one if-statement. The one flag that did hold up, `declining_with_demand`, is safe to keep trusting for now. The team should treat `days_since_last_update`-only staleness rules with real skepticism until paired with something else — on its own it's not predictive of CTR at all.

In [ ]:
print("summary of verdicts:")
print("Test 1 (position -> ctr): MIXED, r =", round(q[["avg_position", "ctr"]].corr().iloc[0, 1], 3))
print("Test 2 (word_count -> engagement): OPPOSITE")
print("Test 3 (staleness -> ctr): FALSE, r =", round(q[["days_since_last_update", "ctr"]].corr().iloc[0, 1], 3))
print("Flag test (declining_with_demand): assumption CONFIRMED (80.9% vs 73.4% base rate)")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.